# Tensor Autograd — Step-by-Step

This notebook builds and explains the `Tensor` autograd engine in
`src/engine.py` from first principles, the same way `Automatic-Gradient.ipynb`
did for the scalar `Value` engine in [ann-foundation](https://github.com/rahulkp-ai/ann-foundation).

The core idea doesn't change when we move from scalars to arrays — it's
still **reverse-mode automatic differentiation via a topologically sorted
computation graph** — but two things become genuinely new and worth
understanding carefully:

1. **Broadcasting-aware gradients.** NumPy silently broadcasts shapes
   (e.g. adding a `(1, n)` bias to a `(batch, n)` matrix). The backward
   pass must "undo" that broadcast by summing gradients back to the
   original shape.
2. **`im2col`-based convolution.** Implementing 2D convolution efficiently
   means unrolling image patches into a matrix so that convolution becomes
   a single matmul — this is what makes a from-scratch CNN actually fast
   enough to train.

We'll build both up gradually, with a numerical gradient check after each
new piece of machinery — never trust an analytical gradient without proof.

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
from src.engine import Tensor, conv2d, max_pool2d

np.random.seed(42)
print('NumPy version:', np.__version__)

NumPy version: 2.0.2


## 2. The `Tensor` class — same skeleton as `Value`, lifted to arrays

Every `Tensor` wraps a NumPy array (`data`) and a same-shaped gradient buffer
(`grad`). Operations record their inputs (`_prev`) and a `_backward` closure
implementing the *local* derivative. `.backward()` does the same two things
`Value.backward()` did:

1. Build a topological ordering of the computation graph (DFS).
2. Walk that order in reverse, calling each node's `_backward()`, which
   accumulates gradients into its parents via the chain rule.

Let's reproduce the exact example from the ann-foundation README, just to
confirm the scalar case still works identically:

In [10]:
a = Tensor(2.0)
b = Tensor(3.0)
c = a * b + a       # c = 8.0
c.backward()

print('c.data  =', c.data, ' (expected 8.0)')
print('a.grad  =', a.grad, ' (expected 4.0, since dc/da = b + 1)')
print('b.grad  =', b.grad, ' (expected 2.0, since dc/db = a)')

c.data  = 8.0  (expected 8.0)
a.grad  = 4.0  (expected 4.0, since dc/da = b + 1)
b.grad  = 2.0  (expected 2.0, since dc/db = a)


## 3. Broadcasting-aware backward

This is the first genuinely new mechanism vs. a scalar engine. Consider
adding a bias vector to every row of a batch:

In [3]:
X = Tensor(np.random.randn(4, 5))     # a "batch" of 4 rows, 5 features each
bias = Tensor(np.random.randn(1, 5))    # one bias value per feature, shared across the batch

out = (X + bias).sum()
out.backward()

print('X.shape    =', X.shape,    '| X.grad.shape    =', X.grad.shape)
print('bias.shape =', bias.shape, '| bias.grad.shape =', bias.grad.shape)

X.shape    = (4, 5) | X.grad.shape    = (4, 5)
bias.shape = (1, 5) | bias.grad.shape = (1, 5)


Notice `bias.grad` has shape `(1, 5)`, matching `bias.data` — even though
the gradient flowing out of `+` was shape `(4, 5)` (the broadcasted output
shape). Internally, `_sum_to_shape()` reduces it by summing over the
broadcasted batch dimension. This makes sense intuitively: each bias value
was reused across all 4 rows, so its gradient is the *sum* of its
contribution to each of those 4 rows (multivariate chain rule).

## 4. Verifying gradients numerically

We never trust an analytical gradient without checking it against a
numerical approximation (central difference). This is the same discipline
`test_engine.py` (in ann-foundation and here) is built around.

In [17]:
def numerical_grad(f, x_data, h=1e-5):
    """Central-difference numerical gradient of scalar function f w.r.t. x_data."""
    grad = np.zeros_like(x_data)
    it = np.nditer(x_data, flags=['multi_index'])
    for _ in it:
        idx = it.multi_index
        orig = x_data[idx]
        x_data[idx] = orig + h
        f_plus = f()
        x_data[idx] = orig - h
        f_minus = f()
        x_data[idx] = orig
        grad[idx] = (f_plus - f_minus) / (2 * h)
    return grad

x_data = np.random.randn(3, 4)

def forward():
    x = Tensor(x_data.copy())
    return (x.relu() ** 2).sum().data.item()

x = Tensor(x_data.copy())
out = (x.relu() ** 2).sum()
out.backward()

num = numerical_grad(forward, x_data)
analytical_grad = 2 * np.maximum(0, x_data)
print("Analytical Gradient (2 * ReLU(x)):\n", analytical_grad)
print("\nNumerical Gradient:\n", num)
print('\n')
print('max abs difference (analytical vs numerical):', np.max(np.abs(x.grad - num)))

Analytical Gradient (2 * ReLU(x)):
 [[3.28993543 0.         1.15311393 0.62250031]
 [6.15776162 2.23914982 0.         0.        ]
 [0.         0.40692727 0.         0.        ]]

Numerical Gradient:
 [[3.28993543 0.         1.15311393 0.62250031]
 [6.15776162 2.23914982 0.         0.        ]
 [0.         0.40692727 0.         0.        ]]


max abs difference (analytical vs numerical): 5.454214857536499e-11


## 5. `im2col`: the trick that makes convolution fast

A naive convolution loops in pure Python over every output pixel and every
kernel position — far too slow to train anything real. The standard trick
(`im2col`) unrolls every receptive-field patch of the image into a column
of a big matrix, so convolution becomes one matrix multiply:

```
conv(input, kernel)  ==  kernel_matrix @ im2col(input)
```

Let's see it in action on a tiny example we can inspect by hand.

In [18]:
from src.engine import _im2col

# A single 1x4x4 "image" with values 0..15, for easy visual inspection
img = np.arange(16, dtype=np.float64).reshape(1, 1, 4, 4)
print('Input image:')
print(img[0, 0])

cols = _im2col(img, kh=3, kw=3, stride=1, pad=0)
print()
print('im2col output shape:', cols.shape, '  (C*kh*kw=9 rows, N*out_h*out_w=4 columns since 4x4 with 3x3 kernel, stride 1 -> 2x2 output)')
print(cols)

Input image:
[[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]
 [12. 13. 14. 15.]]

im2col output shape: (9, 4)   (C*kh*kw=9 rows, N*out_h*out_w=4 columns since 4x4 with 3x3 kernel, stride 1 -> 2x2 output)
[[ 0.  1.  4.  5.]
 [ 1.  2.  5.  6.]
 [ 2.  3.  6.  7.]
 [ 4.  5.  8.  9.]
 [ 5.  6.  9. 10.]
 [ 6.  7. 10. 11.]
 [ 8.  9. 12. 13.]
 [ 9. 10. 13. 14.]
 [10. 11. 14. 15.]]


Each **column** above is one flattened 3x3 receptive field of the input
image. With 4 valid 3x3 windows in a 4x4 image (top-left, top-right,
bottom-left, bottom-right), we get 4 columns. A convolution is now just
`weight.reshape(C_out, -1) @ cols` — a single matmul, whose backward pass
is the *already-verified* matmul backward, composed with `col2im` (the
exact inverse scatter-add operation) to route gradients back to the
original image.

## 6. `conv2d` end-to-end, with a gradient check

In [20]:
x_data = np.random.randn(2, 2, 6, 6) * 0.5
w_data = np.random.randn(3, 2, 3, 3) * 0.5
b_data = np.random.randn(3) * 0.5

def forward():
    x = Tensor(x_data.copy())
    w = Tensor(w_data.copy())
    b = Tensor(b_data.copy())
    return conv2d(x, w, b, stride=1, pad=1).sum().data.item()

x = Tensor(x_data.copy())
w = Tensor(w_data.copy())
b = Tensor(b_data.copy())
out = conv2d(x, w, b, stride=1, pad=1)
print('conv2d output shape:', out.shape, ' (padding=1 keeps spatial size at 6x6)')

out.sum().backward()

num_x = numerical_grad(forward, x_data)
num_w = numerical_grad(forward, w_data)
num_b = numerical_grad(forward, b_data)

print('max abs diff, dx:', np.max(np.abs(x.grad - num_x)))
print('max abs diff, dw:', np.max(np.abs(w.grad - num_w)))
print('max abs diff, db:', np.max(np.abs(b.grad - num_b)))

conv2d output shape: (2, 3, 6, 6)  (padding=1 keeps spatial size at 6x6)
max abs diff, dx: 1.6286998416603637e-09
max abs diff, dw: 1.167885343988928e-09
max abs diff, db: 5.941274139331654e-10


## 7. `max_pool2d`: gradient routes only to the max

Max pooling's backward pass is conceptually simple but easy to implement
incorrectly: the gradient should flow *only* to whichever position was the
maximum in each pooling window — every other position contributes zero
gradient, since an infinitesimal change to a non-max value doesn't change
the max (until it would overtake it, which we ignore as a measure-zero edge
case in continuous gradients).

In [21]:
# A single 2x2 window where the max is unambiguous (4.0, bottom-right)
x = Tensor(np.array([[[[1.0, 2.0], [3.0, 4.0]]]]))
out = max_pool2d(x, pool_size=2, stride=2)
out.sum().backward()

print('Input window:')
print(x.data[0, 0])
print()
print('Gradient (should be 1.0 ONLY at the max position, (1,1)):')
print(x.grad[0, 0])

Input window:
[[1. 2.]
 [3. 4.]]

Gradient (should be 1.0 ONLY at the max position, (1,1)):
[[0. 0.]
 [0. 1.]]


## 8. Putting it together: a tiny conv -> relu -> pool pipeline

This is the exact composition used inside the `CNN` model class in
`src/model.py`, just unrolled here so every step is visible.

In [8]:
x = Tensor(np.random.randn(2, 1, 8, 8) * 0.3)
w = Tensor(np.random.randn(4, 1, 3, 3) * 0.3)
b = Tensor(np.zeros(4))

h = conv2d(x, w, b, stride=1, pad=1)
print('after conv2d:    ', h.shape)
h = h.relu()
print('after relu:      ', h.shape)
h = max_pool2d(h, pool_size=2, stride=2)
print('after max_pool2d:', h.shape)

loss = h.sum()
loss.backward()
print()
print('Gradients are finite and nonzero:')
print('  x.grad finite:', np.all(np.isfinite(x.grad)), '| nonzero:', np.any(x.grad != 0))
print('  w.grad finite:', np.all(np.isfinite(w.grad)), '| nonzero:', np.any(w.grad != 0))

after conv2d:     (2, 4, 8, 8)
after relu:       (2, 4, 8, 8)
after max_pool2d: (2, 4, 4, 4)

Gradients are finite and nonzero:
  x.grad finite: True | nonzero: True
  w.grad finite: True | nonzero: True


## Next steps

This notebook covered the autograd engine in isolation. For the full CNN —
stateful layers, optimizers, and a real MNIST training run with
visualizations — see `examples/MNIST-Training-Visualization.ipynb`.